# Lab | Data Structuring and Combining Data

## Challenge 1: Combining & Cleaning Data

In this challenge, we will be working with the customer data from an insurance company, as we did in the two previous labs. The data can be found here:
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv

But this time, we got new data, which can be found in the following 2 CSV files located at the links below.

- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv

Note that you'll need to clean and format the new data.

Observation:
- One option is to first combine the three datasets and then apply the cleaning function to the new combined dataset
- Another option would be to read the clean file you saved in the previous lab, and just clean the two new files and concatenate the three clean datasets

In [1]:
import pandas as pd

# 1. Load the three raw files
BASE_URL = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/"
raw_files = {name: pd.read_csv(f"{BASE_URL}{name}.csv") for name in ["file1", "file2", "file3"]}

for name, raw in raw_files.items():
    print(f"{name}: {raw.shape}")
    print("   ", raw.columns.tolist())

file1: (4008, 11)
    ['Customer', 'ST', 'GENDER', 'Education', 'Customer Lifetime Value', 'Income', 'Monthly Premium Auto', 'Number of Open Complaints', 'Policy Type', 'Vehicle Class', 'Total Claim Amount']
file2: (996, 11)
    ['Customer', 'ST', 'GENDER', 'Education', 'Customer Lifetime Value', 'Income', 'Monthly Premium Auto', 'Number of Open Complaints', 'Total Claim Amount', 'Policy Type', 'Vehicle Class']
file3: (7070, 11)
    ['Customer', 'State', 'Customer Lifetime Value', 'Education', 'Gender', 'Income', 'Monthly Premium Auto', 'Number of Open Complaints', 'Policy Type', 'Total Claim Amount', 'Vehicle Class']


In [2]:
# 2. One cleaning function that works on any of the three files
def clean_data(df):
    """Standardize the column names, values and data types of one raw customer file."""
    df = df.copy()

    # Column names: lower case, spaces -> "_", "st" -> "state"
    # (file1/file2 call it "ST", file3 calls it "State"; GENDER and Gender both become "gender")
    df.columns = df.columns.str.lower().str.replace(" ", "_", regex=False)
    df = df.rename(columns={"st": "state"})

    # file1 ends with completely empty rows: drop them
    df = df.dropna(how="all").copy()

    # Inconsistent values
    df["gender"] = df["gender"].replace({"Male": "M", "female": "F", "Femal": "F"})
    df["state"] = df["state"].replace({"AZ": "Arizona", "Cali": "California", "WA": "Washington"})
    df["education"] = df["education"].replace({"Bachelors": "Bachelor"})
    df["vehicle_class"] = df["vehicle_class"].replace(
        {"Sports Car": "Luxury", "Luxury SUV": "Luxury", "Luxury Car": "Luxury"}
    )

    # Customer lifetime value: file1/file2 store it as text such as "536307.65%".
    # file3 stores the same customer as 5363.0765, so the "%" means x100. We remove the
    # "%" and divide by 100 so all three files end up on the same scale.
    if not pd.api.types.is_numeric_dtype(df["customer_lifetime_value"]):
        clv = df["customer_lifetime_value"].str.replace("%", "", regex=False)
        df["customer_lifetime_value"] = pd.to_numeric(clv) / 100

    # Number of open complaints: file1/file2 store "1/5/00" -> keep the middle value (5)
    if not pd.api.types.is_numeric_dtype(df["number_of_open_complaints"]):
        df["number_of_open_complaints"] = pd.to_numeric(
            df["number_of_open_complaints"].str.split("/").str[1]
        )

    return df

In [3]:
# 3. Clean each file, then combine them
cleaned = {name: clean_data(raw) for name, raw in raw_files.items()}

# Check: customer lifetime value should now be on the same scale in all three files
for name, d in cleaned.items():
    print(f"{name}: {len(d):>5} rows | mean customer lifetime value = {d['customer_lifetime_value'].mean():,.0f}")

combined = pd.concat(cleaned.values(), ignore_index=True)
print("\nCombined shape:", combined.shape)

# The same customer can appear in more than one file. Keep one row per customer ID,
# taking the first non-missing value of each column (so a missing value in one file
# can be completed from another).
print("Customer IDs that appear more than once:", combined["customer"].duplicated().sum())
combined = combined.groupby("customer", sort=False, as_index=False).first()
print("Shape after keeping one row per customer:", combined.shape)

file1:  1071 rows | mean customer lifetime value = 7,937
file2:   996 rows | mean customer lifetime value = 7,651
file3:  7070 rows | mean customer lifetime value = 8,029

Combined shape: (9137, 11)
Customer IDs that appear more than once: 81
Shape after keeping one row per customer: (9056, 11)


In [4]:
# 4. Null values
print(combined.isnull().sum())

# gender (categorical): "Unknown" instead of guessing; customer lifetime value (skewed): median
combined["gender"] = combined["gender"].fillna("Unknown")
combined["customer_lifetime_value"] = combined["customer_lifetime_value"].fillna(
    combined["customer_lifetime_value"].median()
)
print("\nTotal nulls left:", combined.isnull().sum().sum())

# 5. Numeric variables -> integers (as in the previous lab)
numeric_cols = combined.select_dtypes(include="number").columns
combined[numeric_cols] = combined[numeric_cols].round().astype(int)

# 6. Duplicates check, tidy index and save
print("Duplicate rows:", combined.duplicated().sum(),
      "| duplicate customer IDs:", combined["customer"].duplicated().sum())
combined = combined.reset_index(drop=True)
combined.to_csv("customer_data_combined_clean.csv", index=False)

print("\nFinal shape:", combined.shape)
print(combined.dtypes.to_string())
combined.head()

customer                       0
state                          0
gender                       122
education                      0
customer_lifetime_value        3
income                         0
monthly_premium_auto           0
number_of_open_complaints      0
policy_type                    0
vehicle_class                  0
total_claim_amount             0
dtype: int64

Total nulls left: 0
Duplicate rows: 0 | duplicate customer IDs: 0

Final shape: (9056, 11)
customer                       str
state                          str
gender                         str
education                      str
customer_lifetime_value      int64
income                       int64
monthly_premium_auto         int64
number_of_open_complaints    int64
policy_type                    str
vehicle_class                  str
total_claim_amount           int64


,customer,state,gender,education,customer_lifetime_value,income,monthly_premium_auto,number_of_open_complaints,policy_type,vehicle_class,total_claim_amount
0,RB50392,Washington,Unknown,Master,5780,0,1000,0,Personal Auto,Four-Door Car,3
1,QZ44356,Arizona,F,Bachelor,6980,0,94,0,Personal Auto,Four-Door Car,1131
2,AI49188,Nevada,F,Bachelor,12887,48767,108,0,Personal Auto,Two-Door Car,566
3,WW63253,California,M,Bachelor,7646,0,106,0,Corporate Auto,SUV,530
4,GA49547,Washington,M,High School or Below,5363,36357,68,0,Personal Auto,Four-Door Car,17


**Decisions made in Challenge 1**

- **Option used:** the three raw files are cleaned with the same function and then concatenated (the second option in the observation above).
- **Column names:** the files disagree (`ST` vs `State`, `GENDER` vs `Gender`, different column order), so names are standardized *before* combining. `pd.concat` then lines the columns up by name.
- **Customer lifetime value:** `536307.65%` in file1 is the same number as `5363.0765` for that customer in file3. Only removing the `%` would leave file1 and file2 about 100 times larger than file3 and distort any average of the combined data, so the values are also divided by 100.
- **Repeated customers:** 81 customer IDs appear in more than one file. They are the same customers (same income, claim amount, etc.), so one row per customer ID is kept.
- **Nulls:** blank rows are dropped, missing `gender` becomes `"Unknown"`, and the few missing customer lifetime values get the median, which is not pulled up by very large values.

# Challenge 2: Structuring Data

In this challenge, we will continue to work with customer data from an insurance company, but we will use a dataset with more columns, called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by performing data cleaning, formatting, and structuring.

In [5]:
import pandas as pd

marketing_url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv"
marketing_df = pd.read_csv(marketing_url)

print("Shape:", marketing_df.shape)
marketing_df.head()

Shape: (10910, 27)


,unnamed:_0,customer,state,customer_lifetime_value,response,coverage,education,effective_to_date,employmentstatus,gender,...,number_of_policies,policy_type,policy,renew_offer_type,sales_channel,total_claim_amount,vehicle_class,vehicle_size,vehicle_type,month
0,0,DK49336,Arizona,4809.216960,No,Basic,College,2011-02-18,Employed,M,...,9,Corporate Auto,Corporate L3,Offer3,Agent,292.800000,Four-Door Car,Medsize,A,2
1,1,KX64629,California,2228.525238,No,Basic,College,2011-01-18,Unemployed,F,...,1,Personal Auto,Personal L3,Offer4,Call Center,744.924331,Four-Door Car,Medsize,A,1
2,2,LZ68649,Washington,14947.917300,No,Basic,Bachelor,2011-02-10,Employed,M,...,2,Personal Auto,Personal L3,Offer3,Call Center,480.000000,SUV,Medsize,A,2
3,3,XL78013,Oregon,22332.439460,Yes,Extended,College,2011-01-11,Employed,M,...,2,Corporate Auto,Corporate L3,Offer2,Branch,484.013411,Four-Door Car,Medsize,A,1
4,4,QA50777,Oregon,9025.067525,No,Premium,Bachelor,2011-01-17,Medical Leave,F,...,7,Personal Auto,Personal L2,Offer1,Branch,707.925645,Four-Door Car,Medsize,A,1


1. You work at the marketing department and you want to know which sales channel brought the most sales in terms of total revenue. Using pivot, create a summary table showing the total revenue for each sales channel (branch, call center, web, and mail).
Round the total revenue to 2 decimal points.  Analyze the resulting table to draw insights.

In [6]:
# Total revenue per sales channel (this dataset has no revenue column, so total_claim_amount is used)
pivot1 = (
    marketing_df.pivot_table(
        values="total_claim_amount",
        index="sales_channel",
        aggfunc="sum",
    )
    .rename(columns={"total_claim_amount": "total_revenue"})
    .round(2)
    .sort_values("total_revenue", ascending=False)
)

pivot1

,total_revenue
sales_channel,
Agent,1810226.82
Branch,1301204.00
Call Center,926600.82
Web,706600.04


In [7]:
# Supporting numbers for the analysis: share of the total, number of records and average per record
channel_summary = marketing_df.groupby("sales_channel")["total_claim_amount"].agg(
    records="count", average_per_record="mean"
)
channel_summary["share_of_total_%"] = pivot1["total_revenue"] / pivot1["total_revenue"].sum() * 100
channel_summary.loc[pivot1.index].round(2)

,records,average_per_record,share_of_total_%
sales_channel,,,
Agent,4121,439.27,38.15
Branch,3022,430.58,27.42
Call Center,2141,432.79,19.53
Web,1626,434.56,14.89


**Analysis.** Agent is the strongest channel by a wide margin: about 1.81 million in total revenue, 38.2% of the 4.74 million overall. Branch follows (1.30 million, 27.4%), then Call Center (0.93 million, 19.5%) and Web (0.71 million, 14.9%).

The supporting table shows that the average amount per record is almost the same in every channel (between 431 and 439), so the ranking comes from **volume**, not from higher-value customers: Agent handles 4,121 records against 1,626 for Web. To grow revenue, the levers are more customers per channel rather than richer customers in a particular one.

Two caveats: the dataset has no separate revenue column, so, as the lab does, `total_claim_amount` stands in for revenue; and the channels present in the data are Agent, Branch, Call Center and Web (there is no Mail channel).

2. Create a pivot table that shows the average customer lifetime value per gender and education level. Analyze the resulting table to draw insights.

In [8]:
# Average customer lifetime value per education level and gender
# (margins=True adds the overall average for each row and column)
pivot2 = marketing_df.pivot_table(
    values="customer_lifetime_value",
    index="education",
    columns="gender",
    aggfunc="mean",
    margins=True,
    margins_name="Overall",
).round(2)

pivot2

gender,F,M,Overall
education,,,
Bachelor,7874.27,7703.60,7792.27
College,7748.82,8052.46,7900.07
Doctor,7328.51,7415.33,7372.03
High School or Below,8675.22,8149.69,8415.29
Master,8157.05,8168.83,8162.52
Overall,8071.11,7963.04,8018.24


**Analysis.** Gender alone tells us little: the overall average customer lifetime value is about 8,071 for women and 7,963 for men, a gap of roughly 1.4%.

Education shows a bit more spread. Customers with *High School or Below* have the highest average (about 8,415), followed by *Master* (8,163) and *College* (7,900), while *Doctor* is the lowest (7,372). Higher education does not mean higher lifetime value.

The direction of the gender gap also changes with education: women lead among *High School or Below* (8,675 vs 8,150, about 6%) and *Bachelor* (7,874 vs 7,704), men lead among *College* (8,052 vs 7,749, about 4%), and *Master* and *Doctor* are almost equal. Because there is no consistent pattern, gender-specific targeting is not supported by this table.

These are plain averages: the *Doctor* group is small (about 200 customers per gender), and we have not tested whether any of the differences are statistically significant.

## Bonus

You work at the customer service department and you want to know which months had the highest number of complaints by policy type category. Create a summary table showing the number of complaints by policy type and month.
Show it in a long format table.

*In data analysis, a long format table is a way of structuring data in which each observation or measurement is stored in a separate row of the table. The key characteristic of a long format table is that each column represents a single variable, and each row represents a single observation of that variable.*

*More information about long and wide format tables here: https://www.statology.org/long-vs-wide-data/*

**Approach for the bonus**

- **Sum, not count.** Each row is one customer and `number_of_open_complaints` already holds how many complaints that customer has (0 to 5). Counting rows would count customers with zero complaints as complaints, so the number of complaints is the **sum** of that column.
- **Placeholder values.** The column also contains `0.384256` in hundreds of rows. That is not a possible number of complaints: it is a placeholder (a mean) left over from filling missing values earlier. Summing it would add complaints that do not exist, so those rows are treated as missing and left out of the sum.
- **Month order.** The tables are sorted by month *number* and the names are added afterwards, so the months stay in calendar order (January before February) instead of alphabetical order.

In [9]:
month_names = {1: "January", 2: "February", 3: "March", 4: "April", 5: "May", 6: "June",
               7: "July", 8: "August", 9: "September", 10: "October", 11: "November", 12: "December"}

# 1. How is the complaints column encoded?
complaints = marketing_df["number_of_open_complaints"]
print(complaints.value_counts().sort_index())

# 2. Keep only whole numbers (real complaint counts); the placeholder becomes missing
complaints_df = marketing_df.assign(complaints=complaints.where(complaints % 1 == 0))

# 3. Wide table: sum of complaints by policy type x month
pivot_complaints = complaints_df.pivot_table(
    values="complaints",
    index="policy_type",
    columns="month",
    aggfunc="sum",
)

# 4. Long format: one row per (policy type, month), sorted by month number then policy type
long_df = (
    pivot_complaints.reset_index()
    .melt(id_vars="policy_type", var_name="month", value_name="number_of_complaints")
    .sort_values(["month", "policy_type"])
)
long_df["number_of_complaints"] = long_df["number_of_complaints"].astype(int)
long_df["month"] = long_df["month"].map(month_names)
long_df = long_df.reset_index(drop=True)

print("\nNumber of complaints by policy type and month (long format):")
print(long_df.to_string(index=False))

# 5. Peak month for each policy type, and the number of customer records per month for context
print("\nMonth with the most complaints, per policy type:")
print(long_df.loc[long_df.groupby("policy_type")["number_of_complaints"].idxmax()].to_string(index=False))

print("\nCustomer records per month:")
print(marketing_df["month"].map(month_names).value_counts().reindex(list(long_df["month"].unique())))

number_of_open_complaints
0.000000    8160
0.384256     633
1.000000    1145
2.000000     414
3.000000     324
4.000000     166
5.000000      68
Name: count, dtype: int64

Number of complaints by policy type and month (long format):
   policy_type    month  number_of_complaints
Corporate Auto  January                   415
 Personal Auto  January                  1635
  Special Auto  January                    84
Corporate Auto February                   361
 Personal Auto February                  1363
  Special Auto February                    91

Month with the most complaints, per policy type:
   policy_type    month  number_of_complaints
Corporate Auto  January                   415
 Personal Auto  January                  1635
  Special Auto February                    91

Customer records per month:


month
January     5818
February    5092
Name: count, dtype: int64


**Analysis.** The data only covers January and February, so those are the only months we can compare. January has more complaints for **Personal Auto** (1,635 vs 1,363) and **Corporate Auto** (415 vs 361). **Special Auto** is the only policy type where February is higher (91 vs 84), but the numbers are so small that this is weak evidence.

Personal Auto produces about three quarters of all complaints (2,998 of 3,949), which mirrors its share of the customer base, so it is the biggest source of complaints simply because it is the biggest policy type. January also has more customer records than February (5,818 vs 5,092), so most of January's higher totals reflect more customers, not a worse experience per customer.

Complaints per customer would be a fairer measure than raw totals for deciding where to act; with only two months of data, no seasonal pattern can be claimed.